# Training an LLM Specifically for Proof of Thought

In [18]:
#!/usr/bin/env python
# coding: utf-8

# Advanced RAG on Hugging Face documentation using LangChain (with some packages removed because I only need the generation part of RAG)
# taken from https://huggingface.co/learn/cookbook/en/advanced_rag
# and imports also from https://python.langchain.com/v0.2/docs/integrations/document_loaders/url/
# and imports also from the benchmark/benchmark_full.ipynb

import json
import logging
import time
import sys
import os
from pathlib import Path
from typing import Literal, Any, Optional, List, Tuple
from transformers import AutoModelForCausalLM, AutoTokenizer
import huggingface_hub
import transformers
import torch

# Basically the IPython Magic version of these Linux commands for installations
# cd ..
# pip install -r requirements.txt
# pip install -e .
path_words = os.getcwd().split("/")
if path_words[len(path_words)-1] == "benchmark":
    %cd ..
%pip install -r requirements.txt
%pip install -e .

# Add parent directory to path for z3adapter imports
sys.path.insert(0, str(Path(os.getcwd()).parent.parent))

from utils.azure_config import get_client_config
from z3adapter.reasoning import EvaluationPipeline, ProofOfThought
from z3adapter.reasoning.smt2_prompt_template import build_smt2_prompt
from openai import OpenAI

# Configure logging (output messages during the benchmark). This stores all logs in the outputLog.txt file.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("outputLog.txt", mode="a"),
        logging.StreamHandler(),
    ],
)

print('done importing')

Note: you may need to restart the kernel to use updated packages.
Obtaining file:///home/palindrome868/proofofthought/proofofthought
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for proofofthought (pyproject.toml) ... done
  Created wheel for proofofthought: filename=proofofthought-1.0.1-0.editable-py3-none-any.whl size=8355 sha256=6acfd213cdbe03f3f1849f97310fdd5db2534b37d4c5ec40c1946c3c7f0751de
  Stored in directory: /tmp/pip-ephem-wheel-cache-1rbp5kxz/wheels/00/c1/96/010f9f820116ba3a139dc1cd07e5654713e6181b6544d4cf84
Successfully built proofofthought
  Attempting uninstall: proofofthought
    Found existing installation: proofofthought 1.0.1
    Uninstalling proofofthought-1.0.1:
      Successfully uninstalled proofofthought-1.0.1
Note: you may need to restart the kernel to use updated packages.
done imp

In [19]:
# User-set variables
! hf auth login --token blank  # Your Hug Face authorization token
DATASET_NAME = "proofwriter" # Dataset to be processed
max_samples = 5 # Max number of samples to be processed. Set to 0 to run all samples.
model_name = "Qwen/Qwen3-0.6B" # Name of Hug Face model to be used

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `proof-of-thought-llm-training` has been saved to /home/palindrome868/.cache/huggingface/stored_tokens
Your token has been saved to /home/palindrome868/.cache/huggingface/token
Login successful.
The current active token is: `proof-of-thought-llm-training`


In [22]:
# Set up model, tokenizer, and prompt-processing function
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

def prompt_LLM(prompt, model, tokenizer):
    """Runs a HuggingFace text generation model on the current device.

    model and tokenizer are retrieved from HuggingFace using the model name and the libraries: from transformers import AutoModelForCausalLM, AutoTokenizer.
    
    Input format:
    - prompt: String representing the task for the HuggingFace model (natural language)
    - model: HuggingFace model for answering the prompt
    - tokenizer: for tokenizing input text and un-tokenizing output tokens from the model

    Output format:
    - thinking_content: String representing the thinking process (natural language).
    - content: String representing the response (natural language).
    - time_elapsed: Time this function took to run in ms.
    """

    start_time = time.time()
    
    # Prepare model input
    messages = [ # list of dictionaries
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template( # converts list into text format
        messages,
        tokenize=False,
        add_generation_prompt=True, # So the model knows it should answer the question.
        enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device) # turns text into PyTorch tensors

    # Conduct text completion
    generated_ids = model.generate( # returns a tensor of longs
        **model_inputs, # unpack into separate arguments
        max_new_tokens=65536
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() # [0] to get sequence of tokens for first text. [len(model_inputs.input_ids[0]):] removes original tokens, so only generated ones are kept.

    # Split the tokens based on thinking versus response, then store them as text.
    try:
        # 151668 is the token for </think>, which represents end of thinking content
        index = len(output_ids) - output_ids[::-1].index(151668) # finds the position of value 151668 in the reversed list
    except ValueError: # the case where the model didn't think
        print("No </think> token found.")
        index = 0
    
    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True)
    if thinking_content[0:7] == "<think>":
        thinking_content = thinking_content[8:-9]
    else:
        thinking_content = thinking_content[1:-2]
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True)[2:]

    end_time = time.time()
    time_elapsed = end_time - start_time
    
    # Output
    return thinking_content, content, time_elapsed

# Example usage
prompt = "Give me a short introduction to large language model."
thinking_content, content, time_elapsed = prompt_LLM(prompt, model, tokenizer)
print("thinking content:", thinking_content)
print()
print("content:", content)
print()
print("compute time (seconds): ", time_elapsed)

2026-07-07 18:59:16,119 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-07 18:59:16,135 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/config.json "HTTP/1.1 200 OK"
2026-07-07 18:59:16,230 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-07 18:59:16,247 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-07 18:59:16,357 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-0.6B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-07-07 18:59:16,469 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-0.6B/tree/main?recu

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

2026-07-07 18:59:20,932 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-0.6B/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-07 18:59:20,954 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-0.6B/c1899de289a04d12100db370d81485cdf75e47ca/generation_config.json "HTTP/1.1 200 OK"


thinking content: Okay, the user asked for a short introduction to a large language model. Let me start by recalling what I know about them. Large language models are AI systems designed to understand and generate text, right? They have a huge amount of training data and use complex algorithms to process information.

I should mention that they can understand and generate human-like text, which makes them useful in various fields like writing, content creation, and even customer service. It's important to highlight their ability to handle diverse tasks efficiently. Also, I should keep it concise but informative. Maybe include some key points like training data, algorithms, and applications. Need to make sure it's a short intro without being too technical. Let me check if I'm mixing up anything. Oh, and maybe add that they can be trained on specific tasks or use different types of models. Alright, that should cover it.

content: A large language model (LLM) is an AI system designed to u

In [13]:
# Load Dataset
dataset = None
if DATASET_NAME == "proofwriter":
    dataset = "data/proofwriter_test_processed.json"

dataset_list: list[dict[str, Any]]
if isinstance(dataset, str): # If dataset is a file path
    with open(dataset) as f:
        dataset_list = json.load(f)
else: # If dataset is already a list
    dataset_list = dataset

if isinstance(max_samples, int): # Limit samples if requested
    dataset_list = dataset_list[:max_samples]

for i in range(0, min(len(dataset_list), 5)): # Print first few entries of dataset_list
    print(dataset_list[i])
    print()

{'id': 'RelNeg-OWA-D1-2643(11)', 'question': 'The bald eagle is not big. The bald eagle is blue. The bald eagle likes the mouse. The cow chases the bald eagle. The cow chases the mouse. The cow is round. The cow likes the bald eagle. The cow needs the bald eagle. The cow does not need the mouse. The mouse chases the cow. The mouse is big. The mouse is green. The mouse does not like the bald eagle. The mouse likes the cow. The mouse needs the cow. If something likes the bald eagle and it chases the bald eagle then the bald eagle does not like the cow. If something needs the mouse then it does not like the mouse. If something needs the cow and it is big then it is cold.\n\nQuestion: The mouse is not cold.', 'answer': False, 'QDep': 1, 'original_theory': 'The bald eagle is not big. The bald eagle is blue. The bald eagle likes the mouse. The cow chases the bald eagle. The cow chases the mouse. The cow is round. The cow likes the bald eagle. The cow needs the bald eagle. The cow does not ne

In [25]:
# Run benchmark
for index, sample in enumerate(dataset_list): # Index refers to index in the list, not the id of the question.
    # 1. Prompt the LLM
    prompt = build_smt2_prompt(sample['question'])
    thinking_content, content, time_elapsed = prompt_LLM(prompt, model, tokenizer)
    print("content:", content)
    print()
    
    # 2. Extract the SMT2
    try:
        index = content.index("```") # finds the position of value ``` in the list. the returned index is the first `.
        index2 = len(content) - content[::-1].index("```") # finds the position of value ``` in the reversed list. the returned index is right after the third `.
    except ValueError:
        print("Error: Did not follow the markdown ```smt2 ``` format.")
        index = -3
        index2 = len(content)+3
    extracted_content = content[index+3:index2-3]

    # # 3. 
    # verify_result = self.backend.execute(program_file_path)

    # if not verify_result.success:
    #     error_trace = verify_result.error or "Z3 verification failed"
    #     previous_response = gen_result.raw_response
    #     logger.warning(f"Verification failed: {error_trace}")
    #     continue

    break

KeyboardInterrupt: 

In [ ]:
print("visible CPUs:", len(os.sched_getaffinity(0)))
print("torch threads:", torch.get_num_threads())
print("torch interop threads:", torch.get_num_interop_threads())
print("OMP_NUM_THREADS:", os.getenv("OMP_NUM_THREADS"))
print("MKL_NUM_THREADS:", os.getenv("MKL_NUM_THREADS"))
print("SLURM_CPUS_PER_TASK:", os.getenv("SLURM_CPUS_PER_TASK"))